# Sequence Length Comparison & Generalization tests

In this notebook compare different models at varying sequence lengths and study the length generalization. Additionally, we reproduce a similar version of the causal vs non-causal comparison done in [TabFlex: Scaling Tabular Learning to Millions with Linear Attention](https://arxiv.org/abs/2506.05584).

In [ ]:
NUM_CLASSES = 2
NUM_FEATURES = 20

NUMBER_OF_TEST_SAMPLES = 100
NUMBER_OF_REPETITIONS = 500

########### PATCHING THE CLASS SAMPLER TO ALWAYS RETURN MAX NUM CLASSES ##############

import tabpfn_prior.priors.flexible_categorical as fc

def class_sampler_f(min_, max_): 
    def s():
        return max_
    return s

fc.class_sampler_f = class_sampler_f

############ END PATCHING THE CLASS SAMPLER ##############

import os
import torch
from tqdm import tqdm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pfns.scripts.tabular_metrics import auc_metric

from pfns.priors.tabpfn_prior_adapter import TabPFNPriorConfig
from pfns.priors.prior import Batch
from pfns.train import compute_losses
from pfns.training_utils import move_style_and_check_shape, move_y_style_and_check_shape
from pfns.utils import torch_nanmean, get_default_device
from pfns.scripts.tabpfn_interface import load_model_workflow
from pfns.run_logger import download_model_from_wandb



prior = TabPFNPriorConfig(
    prior_type="mlp",
    max_num_classes=NUM_CLASSES,
    max_num_features=NUM_FEATURES,
    flexible=True,
    differentiable=True,
    return_categorical_mask=True,
    nan_handling=True,
    device="cuda"
)
get_batch = prior.create_get_batch_method()

seqlen_test_list = [200, 400, 600, 800, 1000, 1300, 1600, 2000, 2500, 3000, 4000, 5000, 7000, 10000, 15000, 20000]

device = get_default_device()

models_to_compare = {
    
    # ----------- Transformer Models -----------
    # "Non-Causal_TabPFN": {
    #    "base_path": "./../models_diff/tabpfn_prior_config_1_gpu_v4_masking_0/",
    # },
    # "Causal_TabPFN": {
    #     "base_path": "./../models_diff/tabpfn_prior_config_1_gpu_v4_masking_0_masking_causal_train_only/",
    # },
    
    # ----------- KDA Models -----------
    # "KDA_causal": {
    #     "wandb_run_id": "fla_models/runs/ksmv5v4z",
    # },
    # "KDA_cached": {
    #    "wandb_run_id": "fla_models/runs/qkruutrt",
    # },
    # "KDA_cached_interleaved": {
    #    "wandb_run_id": "fla_models/runs/63y7kc9k",
    # },
    # "KDA_teacher_forcing": {
    #    "wandb_run_id": "fla_models/runs/a925p05n",
    # },
    # "KDA_causal_interleaved": {
    #    "wandb_run_id": "fla_models/runs/cneseyi0",
    # },
    
    # ----------- GLA Models -----------
    # "GLA_Cached": {
    #     "wandb_run_id": "fla_models/runs/g1ul5lyc",
    # },
    # "GLA_Cached_interleaved": {
    #     "wandb_run_id": "fla_models/runs/9k1i2f9z",
    # },
    # "GLA_Causal_interleaved": {
    #     "wandb_run_id": "fla_models/runs/ztdpate1",
    # },
    # "GLA_Teacher_Forcing": {
    #     "wandb_run_id": "fla_models/runs/4f224z23",
    # },
    
    
    # ----------- DeltaNet Models -----------
    # "DeltaNet_Cached_ShortConv": {
    #     "wandb_run_id": "fla_models/runs/nluohjzz", # still running
    # },
    # "DeltaNet_Cached":{
    #     "wandb_run_id": "fla_models/runs/q67a0x92", # still running
    # },
    # "DeltaNet_Teacher_Forcing_ShortConv": {
    #     "wandb_run_id": "fla_models/runs/fm8kzerj", # still running
    # },
    # "DeltaNet_Cached_ShortConv": {
    #     "wandb_run_id": "fla_models/runs/4bvpfdho", # still running
    # },
    # "DeltaNet_Cached": {
    #     "wandb_run_id": "fla_models/runs/ycgd7rq1", # model aocuqz03 lost due to overwrite
    # },
    # "DeltaNet_Teacher_Forcing": {
    #     "wandb_run_id": "fla_models/runs/alqp1bd2", # nupxfnj5 model only at epoch 22 (broken) # seem to have short conv true as no warning of short conv
    # },
    # "DeltaNet_Causal": {
    #     "wandb_run_id": "/fla_models/runs/0bkajhpw", # also has short conv true as no warning of short conv
    # },
    
    # ----------- Gated DeltaNet Models -----------
    "Gated_DeltaNet_Cached_seq_len_10K": {
        "wandb_run_id": "/fla_models/runs/9elhe2fw", # still running (ep 110), can only run on obsession requires 22GB VRAM
    },
    "Gated_DeltaNet_Cached_seq_len_2K": {
        "wandb_run_id": "/fla_models/runs/uah7zywj", # still running (ep 110), can only run on obsession requires 22GB VRAM
    },
    "Gated_DeltaNet_Cached": {
       "wandb_run_id": "/fla_models/runs/abi7ojxu",
    },
    # "Gated_DeltaNet_Teacher_Forcing": {
    #    "wandb_run_id": "/fla_models/runs/16n9ti07", # too old 16n9ti07 and seems broken
    # },
    
    # ----------- Other Models -----------
    
    # "Linear_Attention": {
    #     "base_path": "./../models_diff/linear_attention_config_1_training_setup_high/",
    # },
    # "Rebased": {
    #     "base_path": "./../models_diff/rebased_config_1_training_setup_high/",
    # },
}

models = {}
configs = {}
for model_name, model_config in models_to_compare.items():
    base_path = model_config.get("base_path", ".")
    checkpoint_name = "checkpoint.pt"
    if model_config.get("wandb_run_id"):
        target_path = download_model_from_wandb(
            model_config["wandb_run_id"]
        )
        base_path = os.path.dirname(target_path)
        checkpoint_name = os.path.basename(target_path)
    model, config, _ = load_model_workflow(
        name=checkpoint_name,
        base_path=base_path,
        device=device,
    )
    model.eval()
    models[model_name] = model
    configs[model_name] = config


metric_table = {
    name: {"acc": {}, "ce": {}, "roc_auc": {}} for name in models
}

smallest_seqlen = min(seqlen_test_list)
largest_seqlen = max(seqlen_test_list)

for rep in tqdm(range(NUMBER_OF_REPETITIONS), desc="Overall progress"):
    base_batch = get_batch(
        batch_size=1,
        seq_len=largest_seqlen + NUMBER_OF_TEST_SAMPLES,
        num_features=NUM_FEATURES,
        single_eval_pos=largest_seqlen,
        n_targets_per_input=1,
    )
    
    # ensure train set and eval set have samples from all classes
    unique_classes_in_train = torch.unique(base_batch.y[:, :smallest_seqlen])
    unique_classes_in_eval = torch.unique(base_batch.y[:, largest_seqlen:largest_seqlen + NUMBER_OF_TEST_SAMPLES])
    while (unique_classes_in_train.numel() < NUM_CLASSES) or (unique_classes_in_eval.numel() < NUM_CLASSES):
        base_batch = get_batch(
            batch_size=1,
            seq_len=largest_seqlen + NUMBER_OF_TEST_SAMPLES,
            num_features=NUM_FEATURES,
            single_eval_pos=largest_seqlen,
            n_targets_per_input=1,
        )
        unique_classes_in_train = torch.unique(base_batch.y[:, :smallest_seqlen])
        unique_classes_in_eval = torch.unique(base_batch.y[:, largest_seqlen:largest_seqlen + NUMBER_OF_TEST_SAMPLES])
    
    for model_name, model in models.items():
        config = configs[model_name]

        categorical_inds = None
        if base_batch.categorical_mask is not None:
            mask = base_batch.categorical_mask
            if mask.ndim > 1:
                if not torch.all(mask == mask[0]):
                    raise ValueError("Per-sample categorical masks not supported.")
                mask = mask[0]
            categorical_inds = torch.nonzero(mask, as_tuple=True)[0].tolist()

        with torch.no_grad():
            with torch.autocast(device_type=device, enabled=(model_name in {"DeltaNet_Cached", "DeltaNet_Causal", "DeltaNet_Teacher_Forcing"}), dtype=torch.bfloat16):
                for seqlen_test in seqlen_test_list:
                    x = torch.cat(
                        [
                            base_batch.x[:, :seqlen_test],
                            base_batch.x[:, largest_seqlen : largest_seqlen + NUMBER_OF_TEST_SAMPLES],
                        ],
                        dim=1,
                    )
                    y = torch.cat(
                        [
                            base_batch.y[:, :seqlen_test],
                            base_batch.y[:, largest_seqlen : largest_seqlen + NUMBER_OF_TEST_SAMPLES],
                        ],
                        dim=1,
                    )
                    target_y = torch.cat(
                        [
                            base_batch.target_y[:, :seqlen_test],
                            base_batch.target_y[:, largest_seqlen : largest_seqlen + NUMBER_OF_TEST_SAMPLES],
                        ],
                        dim=1,
                    )

                    test_batch = Batch(
                        x=x,
                        y=y,
                        target_y=target_y,
                        categorical_mask=base_batch.categorical_mask,
                    )

                    output = model(
                        x=test_batch.x.to(device),
                        y=test_batch.y.to(device),
                        style=move_style_and_check_shape(test_batch.style, test_batch.x, device),
                        y_style=move_y_style_and_check_shape(test_batch.y_style, test_batch.y, device),
                        categorical_inds=categorical_inds,
                        only_return_standard_out=True,
                        single_eval_pos=seqlen_test,
                    )

                    targets = test_batch.target_y[:, seqlen_test:].to(device)
                    losses = compute_losses(output, targets, model.criterion, config.n_targets_per_input)
                    loss, _ = torch_nanmean(losses.mean(1), return_nanshare=True)

                    pred = output.argmax(dim=-1)
                    valid = targets != -100
                    if valid.any():
                        accuracy = (pred[valid] == targets[valid]).float().mean().item()
                        
                        probs = torch.softmax(output, dim=-1)
                        auc = auc_metric(targets[valid].cpu(), probs[valid].detach().cpu(), multi_class='ovr')
                        if torch.is_tensor(auc):
                            auc = auc.item()
                    else:
                        print("No valid targets for accuracy/auc computation.")
                        accuracy = float("nan")
                        auc = float("nan")

                    metric_table[model_name]["acc"].setdefault(seqlen_test, []).append(accuracy)
                    metric_table[model_name]["ce"].setdefault(seqlen_test, []).append(float(loss.item()))
                    metric_table[model_name]["roc_auc"].setdefault(seqlen_test, []).append(auc)

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.4)
metrics_to_plot = [
    ("acc", "Accuracy"),
    ("ce", "Cross Entropy Loss"),
    ("roc_auc", "ROC AUC")
]

colors = [
    "#0072B2", "#D55E00", "#009E73", "#B8860B", 
    "#CC79A7", "#56B4E9", "#E69F00", "#000000", 
    "#999999", "#882255", "#44AA99", "#332288"
]
markers = ["o", "s", "D", "^", "v", "<", ">", "p", "*", "X", "h", "8"]
linestyles = [
    "-", "--", "-.", ":", 
    (0, (3, 1, 1, 1)), (0, (5, 1)), 
    (0, (1, 1)), (0, (3, 5, 1, 5)), 
    (0, (3, 1, 1, 1, 1, 1)), (0, (5, 10)),
    "-", "--"
]

legend_converter = {}
for i, model_name in enumerate(models):
    legend_converter[model_name] = (markers[i % len(markers)], linestyles[i % len(linestyles)], colors[i % len(colors)])

lw = 3
markersize = 10

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 6), dpi=300)
fig.subplots_adjust(left=0.05, bottom=0.2, right=0.98, top=0.92, wspace=0.25, hspace=0.3)

for idx, (metric_key, metric_name) in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    for model_name in metric_table:
        if metric_key not in metric_table[model_name] or not metric_table[model_name][metric_key]:
            continue
            
        data = metric_table[model_name][metric_key]
        if not data:
            continue
            
        df = pd.DataFrame(data)
        df_transposed = df.transpose()
        mean = df_transposed.mean(axis=1)
        std = df_transposed.std(axis=1)
        x = df.columns.astype(int)

        sns.lineplot(
            x=x,
            y=mean,
            ax=ax,
            label=model_name if idx == 0 else None,
            linestyle=legend_converter[model_name][1],
            color=legend_converter[model_name][2],
            linewidth=lw,
            marker=legend_converter[model_name][0],
            markersize=markersize,
        )

        # ax.fill_between(x, mean - std, mean + std, alpha=0.3, color=legend_converter[model_name][2])

    ax.set_xlabel("Number of Training Samples")
    ax.set_ylabel(metric_name)
    ax.grid(True, which="both", ls="-", alpha=0.2)
    ax.set_title(f"{metric_name} vs Training Samples")

axes[0].legend(fontsize=12, loc='upper left', bbox_to_anchor=(0, -0.2), ncol=3) 
if axes[1].get_legend(): axes[1].get_legend().remove()
if axes[2].get_legend(): axes[2].get_legend().remove()

plt.show()

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

rows = [
    {"model": m, "metric": k, "seqlen": int(s), "rep": int(r), "value": float(v)}
    for m, metrics in metric_table.items()
    for k, by_len in metrics.items()
    for s, vals in by_len.items()
    for r, v in enumerate(vals)
    if v is not None and not (isinstance(v, float) and np.isnan(v))
]

df = pd.DataFrame(rows)
if df.empty:
    print("No metrics found.")
else:
    df["bucket"] = pd.cut(
        df["seqlen"],
        bins=[0, 500, 1000, 2000, 5000, 10000, float("inf")],
        labels=["≤500", "501-1K", "1K-2K", "2K-5K", "5K-10K", "10K+"],
        include_lowest=True, right=True
    )

    def get_mean_ranks(data, group_cols, rank_scope_cols):
        base = data.groupby(group_cols, observed=True)["value"].mean().reset_index()
        
        is_max_metric = base["metric"].isin(["acc", "roc_auc"])
        base.loc[is_max_metric, "sort_val"] = -base.loc[is_max_metric, "value"]
        base.loc[~is_max_metric, "sort_val"] = base.loc[~is_max_metric, "value"]
        
        base["rank"] = base.groupby(rank_scope_cols, observed=True)["sort_val"].rank(method="average")
        
        final_group = [c for c in group_cols if c != "rep" and c != "value"]
        return base.groupby(final_group, observed=True)["rank"].mean().reset_index()

    overall_ranks = get_mean_ranks(df, ["metric", "rep", "model"], ["metric", "rep"])
    
    bucket_ranks = get_mean_ranks(df, ["metric", "bucket", "rep", "model"], ["metric", "bucket", "rep"])
    
    seqlen_ranks = get_mean_ranks(df, ["metric", "seqlen", "rep", "model"], ["metric", "seqlen", "rep"])

    for metric in sorted(df["metric"].unique()):
        print(f"\n=== Mean rank (lower is better) for {metric} ===")
        
        overall_disp = (
            overall_ranks[overall_ranks["metric"] == metric]
            .sort_values("rank")
            .drop(columns="metric")
            .reset_index(drop=True)
        )
        display(overall_disp.style.format({"rank": "{:.2f}"}).background_gradient(subset=["rank"], cmap="Greens_r"))

        print(f"\n=== Mean rank by bucket for {metric} ===")
        pivot = bucket_ranks[bucket_ranks["metric"] == metric].pivot_table(
            index="model", columns="bucket", values="rank", observed=True
        )
        display(pivot.style.format("{:.2f}").background_gradient(cmap="Greens_r", axis=None))
        
        print(f"\n=== Mean rank by individual sequence length for {metric} ===")
        pivot_seqlen = seqlen_ranks[seqlen_ranks["metric"] == metric].pivot_table(
            index="model", columns="seqlen", values="rank", observed=True
        )
        display(pivot_seqlen.style.format("{:.2f}").background_gradient(cmap="Greens_r", axis=None))

    print("\nNote: rank=1 is best. Rankings are computed per repeat, then averaged.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if "bucket_ranks" in locals() and not bucket_ranks.empty:
    metrics = sorted(bucket_ranks["metric"].unique())
    n_metrics = len(metrics)
    
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 6), dpi=300)
    fig.subplots_adjust(left=0.05, bottom=0.2, right=0.98, top=0.92, wspace=0.0, hspace=0.0)
    if n_metrics == 1:
        axes = [axes]
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        plot_data = bucket_ranks[bucket_ranks["metric"] == metric].copy()
        unique_models = sorted(plot_data["model"].unique())
        
        for model in unique_models:
            subset = plot_data[plot_data["model"] == model]
            
            color = None
            linestyle = "-"
            marker = "o"
            
            if "legend_converter" in locals() and model in legend_converter:
                marker, linestyle, color = legend_converter[model]
            
            sns.lineplot(
                data=subset,
                x="bucket",
                y="rank",
                label=model if idx == 0 else None,
                color=color,
                linestyle=linestyle,
                marker=marker,
                linewidth=3,
                markersize=12,
                errorbar=None,
                ax=ax,
                legend=False
            )
        
        ax.set_title(f"Ranking: {metric}", fontsize=16)
        ax.set_ylabel("Average Rank (1 = Best)", fontsize=14)
        ax.set_xlabel("Sequence Length Bucket", fontsize=14)
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3)
    
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(fontsize=12, loc='upper left', bbox_to_anchor=(0, -0.2), ncol=3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Rankings not found. Please run the previous cell to compute 'bucket_ranks'.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if "seqlen_ranks" in locals() and not seqlen_ranks.empty:
    metrics = sorted(seqlen_ranks["metric"].unique())
    n_metrics = len(metrics)
    
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 6), dpi=300)
    fig.subplots_adjust(left=0.05, bottom=0.2, right=0.98, top=0.92, wspace=0.0, hspace=0.3)
    if n_metrics == 1:
        axes = [axes]
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        plot_data = seqlen_ranks[seqlen_ranks["metric"] == metric].copy()
        unique_models = sorted(plot_data["model"].unique())
        
        for model in unique_models:
            subset = plot_data[plot_data["model"] == model].sort_values("seqlen")
            
            color = None
            linestyle = "-"
            marker = "o"
            
            if "legend_converter" in locals() and model in legend_converter:
                marker, linestyle, color = legend_converter[model]
            
            sns.lineplot(
                data=subset,
                x="seqlen",
                y="rank",
                label=model if idx == 0 else None,
                color=color,
                linestyle=linestyle,
                marker=marker,
                linewidth=3,
                markersize=10,
                errorbar=None,
                ax=ax,
                legend=False
            )
        
        ax.set_title(f"Ranking: {metric}", fontsize=16)
        ax.set_ylabel("Average Rank (1 = Best)", fontsize=14)
        ax.set_xlabel("Sequence Length", fontsize=14)
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
    
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(fontsize=12, loc='upper left', bbox_to_anchor=(0, -0.2), ncol=3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Sequence length rankings not found. Please run the previous cell to compute 'seqlen_ranks'.")